<a href="https://colab.research.google.com/github/Enrik-Shabani/ML-Project-GroupA/blob/main/notebooks/02_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Exploratory Data Analysis (EDA)
## Beijing Multi-Site Air Quality · HEC Lausanne ML Project

**Reproducibility** : This notebook is fully self-contained. The dataset is downloaded automatically from a public Google Drive link. Simply run all cells.

**Input** : `data_clean_featured.csv` (produced by Notebook 1)  
**Target** : `PM2.5`  
**Dataset** : Beijing Multi-Site Air Quality 2013–2017, 12 stations

---
### Table of Contents
1. [Setup & Data Loading](#setup)
2. [Dataset Overview](#overview)
3. [Target Distribution — PM2.5](#target)
4. [Temporal Patterns](#temporal)
5. [Correlations & Inter-feature Relationships](#cprrelations)
6. [Spatial Analysis by Station](#spatial)
7. [Weather Variables × PM2.5](#weather)
8. [Lag & Rolling Features Analysis](#lag)
9. [EDA Summary → Implications for Notebooks 3 & 4](#summary)

---
<a name="setup"></a>
## 1. Setup & Data Loading

In [ ]:
# gdown is pre-installed in Colab; this ensures the latest version
!pip install -q gdown --upgrade

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

# ── Global plot style ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# ── Project color palette ────────────────────────────────────────────────────
PALETTE_SEASONS = {
    'Spring': '#4CAF50',
    'Summer': '#FF9800',
    'Autumn': '#795548',
    'Winter': '#2196F3'
}
COLOR_TARGET = '#E53935'
COLOR_ACCENT = '#1565C0'

print('Libraries loaded.')

In [ ]:
import gdown

# Download dataset from public Google Drive link
FILE_ID     = '1T9gq-MBUJyZHIW8uMIRPzfCxNslLov3j'
OUTPUT_PATH = 'data_clean_featured.csv'

gdown.download(id=FILE_ID, output=OUTPUT_PATH, quiet=False)

df = pd.read_csv(OUTPUT_PATH)
print(f'\nDataset loaded — shape: {df.shape}')
df.head(3)

---
<a name="overview"></a>
## 2. Dataset Overview

In [ ]:
# ── Define column groups ─────────────────────────────────────────────────────
POLLUTANTS      = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
WEATHER         = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
WIND            = [c for c in ['wd_sin','wd_cos','wind_u','wind_v'] if c in df.columns]
TIME_CYCLIC     = [c for c in ['hour_sin','hour_cos','month_sin','month_cos'] if c in df.columns]
LAGS            = [c for c in ['pm25_lag_1h','pm25_lag_3h','pm25_lag_6h'] if c in df.columns]
ROLLING         = [c for c in ['pm25_roll_3h','pm25_roll_6h','pm25_roll_24h'] if c in df.columns]
STATION_DUMMIES = [c for c in df.columns if c.startswith('station_')]

print(f'Pollutants     : {POLLUTANTS}')
print(f'Weather        : {WEATHER}')
print(f'Lag features   : {LAGS}')
print(f'Rolling features: {ROLLING}')
print(f'Station dummies: {STATION_DUMMIES}')

In [ ]:
# ── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().mean().mul(100).round(2)
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print('No missing values — dataset is clean.')
else:
    print('Columns with missing values (%):')
    display(missing.to_frame('null_%'))

In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────────
print('=== Pollutants ===')
display(df[POLLUTANTS].describe().round(2))
print('\n=== Weather variables ===')
display(df[WEATHER].describe().round(2))

In [ ]:
# ── Temporal coverage & station breakdown ────────────────────────────────────
print(f'Years    : {sorted(df["year"].unique())}')
print(f'Total rows: {len(df):,}')

# Detect or reconstruct station column
if 'station' in df.columns:
    STATION_COL = 'station'
elif STATION_DUMMIES:
    df['station'] = (df[STATION_DUMMIES]
                     .idxmax(axis=1)
                     .str.replace('station_', '', regex=False))
    STATION_COL = 'station'
    print('Station column reconstructed from one-hot dummies.')
else:
    STATION_COL = None
    print('Warning: station column not found.')

if STATION_COL:
    display(df[STATION_COL].value_counts().to_frame('observations'))

# Season labels
SEASON_MAP   = {1: 'Spring', 2: 'Summer', 3: 'Autumn', 4: 'Winter'}
SEASON_ORDER = ['Spring', 'Summer', 'Autumn', 'Winter']
df['season_label'] = (
    df['season'].map(SEASON_MAP)
    if df['season'].dtype != object
    else df['season']
)

---
<a name="target"></a>
## 3. Target Distribution — PM2.5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Raw distribution
ax = axes[0]
ax.hist(df['PM2.5'].dropna(), bins=80, color=COLOR_TARGET,
        alpha=0.75, edgecolor='white', linewidth=0.3)
ax.axvline(df['PM2.5'].median(), color='black', lw=1.5, ls='--',
           label=f'Median = {df["PM2.5"].median():.1f}')
ax.axvline(df['PM2.5'].mean(), color='gray', lw=1.5, ls=':',
           label=f'Mean = {df["PM2.5"].mean():.1f}')
ax.set_title('Raw PM2.5 distribution')
ax.set_xlabel('PM2.5 (µg/m³)')
ax.legend(fontsize=9)

# Log-transformed
ax = axes[1]
log_pm25 = np.log1p(df['PM2.5'].dropna())
ax.hist(log_pm25, bins=60, color=COLOR_ACCENT,
        alpha=0.75, edgecolor='white', linewidth=0.3)
ax.set_title(f'log(1 + PM2.5)  |  skew = {log_pm25.skew():.3f}')
ax.set_xlabel('log(1 + PM2.5)')

# QQ-plot
ax = axes[2]
stats.probplot(log_pm25, dist='norm', plot=ax)
ax.set_title('QQ-plot — log(1 + PM2.5)')
ax.get_lines()[0].set(markersize=1.5, alpha=0.3, color=COLOR_ACCENT)
ax.get_lines()[1].set(color='red', lw=1.5)

plt.tight_layout()
plt.savefig('fig_03a_pm25_distribution.png', bbox_inches='tight')
plt.show()

print(f'Raw skewness            : {df["PM2.5"].skew():.3f}')
print(f'Log skewness            : {log_pm25.skew():.3f}')
print(f'% > 150 µg/m³ (very unhealthy) : {(df["PM2.5"] > 150).mean()*100:.1f}%')
print(f'% >  75 µg/m³ (unhealthy)      : {(df["PM2.5"] >  75).mean()*100:.1f}%')
print(f'% >  35 µg/m³ (moderate)       : {(df["PM2.5"] >  35).mean()*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='season_label', y='PM2.5', order=SEASON_ORDER,
            palette=PALETTE_SEASONS, width=0.5, fliersize=1.5,
            flierprops=dict(alpha=0.3), ax=ax)
ax.set_title('PM2.5 distribution by season')
ax.set_xlabel('')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.savefig('fig_03b_pm25_by_season.png', bbox_inches='tight')
plt.show()

display(
    df.groupby('season_label')['PM2.5']
    .agg(['mean','median','std','max'])
    .round(2)
    .loc[SEASON_ORDER]
)

---
<a name="temporal"></a>
## 4. Temporal Patterns

In [ ]:
# Hourly profile — global and by season
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly = df.groupby('hour')['PM2.5'].median()
ax = axes[0]
ax.plot(hourly.index, hourly.values, color=COLOR_TARGET, lw=2, marker='o', markersize=4)
ax.fill_between(hourly.index, hourly.values, alpha=0.12, color=COLOR_TARGET)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Median PM2.5 (µg/m³)')
ax.set_title('Median hourly profile — all stations')
ax.set_xticks(range(0, 24, 2))

ax = axes[1]
for season in SEASON_ORDER:
    sub = df[df['season_label'] == season].groupby('hour')['PM2.5'].median()
    ax.plot(sub.index, sub.values, label=season, color=PALETTE_SEASONS[season], lw=1.8)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Median PM2.5 (µg/m³)')
ax.set_title('Median hourly profile by season')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_04a_hourly_profile.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmaps — hour × day-of-week  and  month × hour
pivot_hd = df.pivot_table(values='PM2.5', index='hour', columns='dayofweek', aggfunc='median')
pivot_hd.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

pivot_mh = df.pivot_table(values='PM2.5', index='month', columns='hour', aggfunc='median')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(pivot_hd, cmap='YlOrRd', ax=axes[0], linewidths=0.3,
            cbar_kws={'label': 'Median PM2.5 (µg/m³)'})
axes[0].set_title('PM2.5 — Hour × Day of week')
axes[0].set_ylabel('Hour')
axes[0].set_xlabel('')

sns.heatmap(pivot_mh, cmap='YlOrRd', ax=axes[1], linewidths=0.2,
            cbar_kws={'label': 'Median PM2.5 (µg/m³)'})
axes[1].set_title('PM2.5 — Month × Hour')
axes[1].set_ylabel('Month')
axes[1].set_xlabel('Hour')

plt.tight_layout()
plt.savefig('fig_04b_heatmaps.png', bbox_inches='tight')
plt.show()

In [ ]:
# Long-term monthly trend 2013–2017
monthly = df.groupby(['year','month'])['PM2.5'].median().reset_index()
monthly['date'] = pd.to_datetime(
    monthly['year'].astype(str) + '-' + monthly['month'].astype(str) + '-01'
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['date'], monthly['PM2.5'], color=COLOR_TARGET, lw=1.5, alpha=0.85)
ax.fill_between(monthly['date'], monthly['PM2.5'], alpha=0.12, color=COLOR_TARGET)
for yr in monthly['year'].unique():
    x = pd.Timestamp(f'{yr}-01-01')
    ax.axvline(x, color='gray', lw=0.7, ls='--', alpha=0.5)
    ax.text(x, monthly['PM2.5'].max() * 0.93, str(yr), fontsize=8, color='gray')
ax.set_title('Monthly PM2.5 trend — median across all stations (2013–2017)')
ax.set_ylabel('Median PM2.5 (µg/m³)')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('fig_04c_monthly_trend.png', bbox_inches='tight')
plt.show()

---
<a name="correlations"></a>
## 5. Correlations & Inter-feature Relationships

In [ ]:
NUMERIC_FEATURES = POLLUTANTS + WEATHER + WIND + TIME_CYCLIC + LAGS + ROLLING
NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f in df.columns]

corr = df[NUMERIC_FEATURES].corr()

fig, ax = plt.subplots(figsize=(17, 13))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, annot=True, fmt='.2f', annot_kws={'size': 7},
    ax=ax, square=True
)
ax.set_title('Pearson correlation matrix — all numeric features', pad=12)
plt.tight_layout()
plt.savefig('fig_05a_correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature ranking by correlation with PM2.5
corr_pm25 = corr['PM2.5'].drop('PM2.5').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
colors = [COLOR_TARGET if v > 0 else COLOR_ACCENT for v in corr_pm25.values]
ax.barh(corr_pm25.index, corr_pm25.values, color=colors, alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson r with PM2.5')
ax.set_title('All features ranked by |correlation| with PM2.5')
ax.invert_yaxis()
for i, (feat, val) in enumerate(corr_pm25.items()):
    ax.text(val + 0.01 * np.sign(val), i, f'{val:.3f}',
            va='center', fontsize=8, ha='left' if val >= 0 else 'right')
plt.tight_layout()
plt.savefig('fig_05b_corr_with_pm25.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots — top 6 most correlated features
top6   = corr_pm25.head(6).index.tolist()
sample = df.sample(min(5000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top6):
    valid = sample[[feat, 'PM2.5']].dropna()
    r     = valid.corr().iloc[0, 1]
    ax.scatter(valid[feat], valid['PM2.5'], alpha=0.15, s=5, color=COLOR_ACCENT)
    m, b = np.polyfit(valid[feat], valid['PM2.5'], 1)
    xs   = np.linspace(valid[feat].min(), valid[feat].max(), 100)
    ax.plot(xs, m * xs + b, color=COLOR_TARGET, lw=1.5)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('PM2.5', fontsize=9)
    ax.set_title(f'r = {r:.3f}', fontsize=10)

plt.suptitle('PM2.5 vs top 6 correlated features', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('fig_05c_scatter_top6.png', bbox_inches='tight')
plt.show()

---
<a name="spatial"></a>
## 6. Spatial Analysis by Station

In [ ]:
if STATION_COL:
    station_stats = (
        df.groupby(STATION_COL)['PM2.5']
        .agg(['mean','median','std','max'])
        .round(2)
        .sort_values('median', ascending=False)
    )
    display(station_stats)

    overall_std      = df['PM2.5'].std()
    inter_station    = station_stats['median'].std()
    intra_station    = station_stats['std'].mean()
    print(f'\nOverall PM2.5 std           : {overall_std:.2f}')
    print(f'Inter-station std (medians) : {inter_station:.2f}')
    print(f'Mean intra-station std      : {intra_station:.2f}')

In [ ]:
if STATION_COL:
    station_order = station_stats.index.tolist()
    cmap = plt.cm.get_cmap('tab20', len(station_order))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Boxplot
    sns.boxplot(data=df, x=STATION_COL, y='PM2.5', order=station_order,
                palette='Blues_r', width=0.55, fliersize=1.5,
                flierprops=dict(alpha=0.2), ax=axes[0])
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
    axes[0].set_title('PM2.5 distribution by station')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('PM2.5 (µg/m³)')

    # Hourly profiles
    ax = axes[1]
    for i, sta in enumerate(station_order):
        sub = df[df[STATION_COL] == sta].groupby('hour')['PM2.5'].median()
        ax.plot(sub.index, sub.values, lw=1.2, alpha=0.8, label=sta, color=cmap(i))
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Median PM2.5 (µg/m³)')
    ax.set_title('Hourly PM2.5 profile by station')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=7, ncol=2, loc='upper left')

    plt.tight_layout()
    plt.savefig('fig_06_spatial_analysis.png', bbox_inches='tight')
    plt.show()

---
<a name="weather"></a>
## 7. Weather Variables × PM2.5

In [ ]:
sample = df.sample(min(8000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feat in zip(axes.flat, ['TEMP', 'PRES', 'DEWP', 'WSPM']):
    valid = sample[[feat, 'PM2.5', 'hour']].dropna()
    r     = valid[[feat, 'PM2.5']].corr().iloc[0, 1]
    sc    = ax.scatter(valid[feat], valid['PM2.5'],
                       c=valid['hour'], cmap='twilight', alpha=0.3, s=6)
    ax.set_xlabel(feat)
    ax.set_ylabel('PM2.5 (µg/m³)')
    ax.set_title(f'{feat} vs PM2.5  |  r = {r:.3f}')

plt.colorbar(sc, ax=axes[1, 1], label='Hour of day')
plt.suptitle('Weather variables vs PM2.5  (color = hour of day)', y=1.01)
plt.tight_layout()
plt.savefig('fig_07a_weather_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# Wind rose — median PM2.5 by wind direction
if 'wd_sin' in df.columns and 'wd_cos' in df.columns:
    df['wd_rad'] = np.arctan2(df['wd_sin'], df['wd_cos'])
    N_BINS       = 16
    bins         = np.linspace(-np.pi, np.pi, N_BINS + 1)

    # Use include_lowest=True to ensure the lowest bin edge is included
    # and ensure categorical output to use with reindex correctly
    df['wd_bin'] = pd.cut(df['wd_rad'], bins=bins, labels=False, include_lowest=True)

    rose_raw     = df.groupby('wd_bin')['PM2.5'].median()

    # Reindex to ensure all bins from 0 to N_BINS-1 are present, filling missing with 0
    # This makes 'rose' have N_BINS elements, matching 'theta'
    rose         = rose_raw.reindex(range(N_BINS), fill_value=0)

    theta        = bins[:-1] + np.diff(bins) / 2
    width        = 2 * np.pi / N_BINS

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(projection='polar'))

    # Ensure norm handles cases where rose_raw might be empty or min/max are the same
    min_val = rose_raw.min() if not rose_raw.empty else 0
    max_val = rose_raw.max() if not rose_raw.empty else 1 # Ensure max_val > min_val if rose_raw is empty
    norm = plt.Normalize(min_val, max_val)

    ax.bar(theta, rose.values, width=width, alpha=0.8, bottom=0,
           color=plt.cm.YlOrRd(norm(rose.values)))
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title('Median PM2.5 by wind direction', pad=15)
    sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Median PM2.5', shrink=0.65, pad=0.1)
    plt.tight_layout()
    plt.savefig('fig_07b_wind_rose.png', bbox_inches='tight')
    plt.show()

In [ ]:
# PM2.5 by wind speed category
df['wspm_cat'] = pd.cut(
    df['WSPM'],
    bins=[-0.1, 1, 2, 3, 5, 100],
    labels=['0–1', '1–2', '2–3', '3–5', '>5']
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='wspm_cat', y='PM2.5', palette='Blues',
            width=0.5, fliersize=1.5, flierprops=dict(alpha=0.2), ax=ax)
ax.set_xlabel('Wind speed WSPM (m/s)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 by wind speed category')
plt.tight_layout()
plt.savefig('fig_07c_wind_speed_pm25.png', bbox_inches='tight')
plt.show()

---
<a name="lag"></a>
## 8. Lag & Rolling Features Analysis

In [ ]:
# Correlation table
lag_roll_all = LAGS + ROLLING
if lag_roll_all:
    corr_lag = (
        df[lag_roll_all + ['PM2.5']]
        .corr()['PM2.5']
        .drop('PM2.5')
        .sort_values(ascending=False)
    )
    display(corr_lag.to_frame('Pearson r').round(4))

In [ ]:
# Scatter PM2.5(t) vs selected lag/rolling features
feats_to_plot = [f for f in ['pm25_lag_1h', 'pm25_lag_6h', 'pm25_roll_24h']
                 if f in df.columns]

if feats_to_plot:
    sample = df.sample(min(6000, len(df)), random_state=42)
    fig, axes = plt.subplots(1, len(feats_to_plot),
                             figsize=(5 * len(feats_to_plot), 4))
    if len(feats_to_plot) == 1:
        axes = [axes]
    for ax, feat in zip(axes, feats_to_plot):
        valid = sample[[feat, 'PM2.5']].dropna()
        r     = valid.corr().iloc[0, 1]
        ax.scatter(valid[feat], valid['PM2.5'], alpha=0.2, s=5, color=COLOR_ACCENT)
        lim = [0, max(valid[feat].max(), valid['PM2.5'].max())]
        ax.plot(lim, lim, 'r--', lw=1, label='y = x')
        ax.set_xlabel(feat)
        ax.set_ylabel('PM2.5 (t)')
        ax.set_title(f'r = {r:.4f}')
        ax.legend(fontsize=8)
    plt.suptitle('PM2.5(t) vs lag & rolling features', y=1.01)
    plt.tight_layout()
    plt.savefig('fig_08a_lag_scatter.png', bbox_inches='tight')
    plt.show()

In [ ]:
# Autocorrelation function — 48 hours
if STATION_COL:
    ref_station  = df[STATION_COL].value_counts().index[0]
    ts           = df[df[STATION_COL] == ref_station]['PM2.5'].dropna()
    title_suffix = f'station: {ref_station}'
else:
    ts           = df['PM2.5'].dropna()
    title_suffix = 'all stations'

MAX_LAG  = 48
ci       = 1.96 / np.sqrt(len(ts))
acf_vals = [ts.autocorr(lag=i) for i in range(1, MAX_LAG + 1)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(1, MAX_LAG + 1), acf_vals, color=COLOR_TARGET, alpha=0.7)
ax.axhline(0,   color='black', lw=0.8)
ax.axhline( ci, color='steelblue', lw=1.2, ls='--', label='95% CI')
ax.axhline(-ci, color='steelblue', lw=1.2, ls='--')
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('Autocorrelation')
ax.set_title(f'PM2.5 autocorrelation function — {title_suffix}')
ax.set_xticks(range(0, MAX_LAG + 1, 6))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_08b_acf.png', bbox_inches='tight')
plt.show()

for lag in [1, 3, 6, 24]:
    print(f'ACF lag-{lag:>2}h : {acf_vals[lag-1]:.4f}')

---
<a name="summary"></a>
## 9. EDA Summary → Implications for Notebooks 3 & 4

In [ ]:
# Full feature ranking
all_numeric = [f for f in NUMERIC_FEATURES if f != 'PM2.5' and f in df.columns]
ranking     = (
    df[all_numeric + ['PM2.5']]
    .corr()['PM2.5']
    .drop('PM2.5')
    .abs()
    .sort_values(ascending=False)
)

print('=== TOP 15 features by |Pearson r| with PM2.5 ===')
display(ranking.head(15).to_frame('|r|').round(4))

print('\n=== BOTTOM 10 features (weakest correlation) ===')
display(ranking.tail(10).to_frame('|r|').round(4))

In [ ]:
# Export recommended features for Notebook 3
df['log_PM2.5'] = np.log1p(df['PM2.5'])

recommended = ranking[ranking > 0.05].index.tolist()

eda_output = {
    'recommended_features': recommended,
    'target'              : 'PM2.5',
    'log_target'          : 'log_PM2.5',
    'notes': {
        'distribution'  : 'Strongly right-skewed; log(1+PM2.5) is approx. Gaussian',
        'seasonality'   : 'Winter >> Spring/Autumn >> Summer',
        'top_predictors': 'Lag/rolling features dominate; PM10, CO, NO2 highly correlated',
        'validation'    : 'Use TimeSeriesSplit — never random train/test split on time series',
        'scaling'       : 'StandardScaler required for Ridge regression',
        'clustering'    : 'K-Means on mean pollutant profiles per station for Notebook 4'
    }
}

with open('eda_output.json', 'w') as f:
    json.dump(eda_output, f, indent=2)

print(f'Recommended features ({len(recommended)}): {recommended}')
print('\nSaved → eda_output.json')
print('Load in Notebook 3:')
print("  import json")
print("  eda      = json.load(open('eda_output.json'))")
print("  features = eda['recommended_features']")

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    EDA SUMMARY — Key Findings                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PM2.5 DISTRIBUTION                                                  ║
║  · Strongly right-skewed — heavy tail of high-pollution episodes     ║
║  · log(1+PM2.5) is approximately Gaussian → prefer as Ridge target   ║
║  · Significant fraction of hours exceeds WHO thresholds              ║
║                                                                      ║
║  TEMPORAL PATTERNS                                                   ║
║  · Two daily peaks: early morning (6–9 AM) and evening (8–11 PM)     ║
║  · Strong seasonality: Winter >> Spring/Autumn >> Summer             ║
║  · hour_sin/cos and month_sin/cos cleanly encode these cycles        ║
║                                                                      ║
║  TOP PREDICTIVE FEATURES (→ Notebook 3)                              ║
║  · pm25_lag_1h / 3h / 6h    — very high autocorrelation              ║
║  · pm25_roll_3h / 24h        — capture persistence                   ║
║  · PM10, CO, NO2             — co-emitted pollutants                 ║
║  · TEMP, DEWP                — atmospheric dispersion conditions     ║
║  · WSPM (negative)           — wind disperses particulate matter     ║
║                                                                      ║
║  SPATIAL ANALYSIS (→ Notebook 4 — Clustering)                        ║
║  · Significant inter-station variability in PM2.5 levels             ║
║  · Hourly profiles are structurally similar across stations          ║
║  · K-Means on mean pollutant levels per station is a natural fit     ║
║                                                                      ║
║  MODELING RECOMMENDATIONS                                            ║
║  · Ridge   : StandardScaler required; use log(PM2.5) as target       ║
║  · RF / GB : lag features dominate; verify no data leakage           ║
║  · Validation : TimeSeriesSplit — never random split on time series  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

## EDA Summary — Implications for ML

- PM2.5 is highly right-skewed, so log transformation may be useful.
- PM2.5 has strong temporal autocorrelation, especially at 1h, 3h and 6h lags.
- Lag and rolling features are likely to be among the strongest predictors.
- Weather variables, especially wind speed, seem relevant because stronger wind is associated with lower PM2.5.
- Station identity matters, so station dummy variables should be kept.
- Because the dataset is temporal, supervised learning should use a chronological split rather than a random split.
- For unsupervised learning, clustering can be applied to pollution profiles by station, hour, season, or weather conditions.